<a href="https://colab.research.google.com/github/alter-mix-dev/wasapcloudapi/blob/main/Copia_de_Hands_On_WhatsApp_Cloud_API_(Soluci%C3%B3n).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: WHATSAPP CLOUD API**

Una vez vista la masterclass ***WhatsApp Cloud API***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

## **LOGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"  # debe coincidir con el configurado en el dashboard de Meta
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')  # visible en API Setup del dashboard de Meta

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALACIÓN Y ARRANQUE DE OLLAMA**

Ollama no viene preinstalado en la máquina virtual de Google Colab, por lo que se debe instalar y arrancar en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd # requiere zstd, que Colab no trae por defecto
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 95 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (335 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [ ]:
# Iniciar el servidor de Ollama

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo apuntando al Ollama ya activo

!pip install llama-stack llama-stack-client --quiet

import subprocess
import time
import os

os.environ["OLLAMA_URL"] = "http://localhost:11434"

proceso_llama_stack = subprocess.Popen(["llama", "stack", "run", "starter", "--port", "8321"])
time.sleep(20)  # tarda mas en levantar que Ollama

from llama_stack_client import LlamaStackClient
client = LlamaStackClient(base_url="http://localhost:8321")
print("Llama Stack listo.")

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

Con Ollama corriendo, se arma la lógica del asistente: la estructura del servidor con sus registros en memoria, la normalización de números mexicanos (necesaria para que la API de envío acepte el número que entrega el webhook), y la función que genera la respuesta real con Llama 3.1.

In [ ]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()  # numeros que ya recibieron el saludo inicial

In [ ]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

In [ ]:
# Definir función de generación de respuesta

def generar_respuesta(texto_usuario):
    prompt = (
        "Eres el asistente de atención a clientes de un gimnasio. "
        "Responde de forma breve (2-3 líneas) a la siguiente pregunta sobre horarios o membresías:\n\n"
        + texto_usuario
    )

    respuesta = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content": prompt}])
    return respuesta["message"]["content"]

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante.

La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta, evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificación de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [ ]:
# Definir endpoint de recepción de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}  # Meta recibe el ACK de inmediato, sin esperar a Ollama

In [ ]:
# Definir función de envío de respuesta

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    requests.post(url, headers=headers, json=payload)

In [ ]:
# Definir función de procesamiento de respuesta

def procesar_y_responder(numero, texto):
    if numero not in usuarios_conocidos:
        usuarios_conocidos.add(numero)
        enviar_respuesta(numero, "¡Hola! Soy el asistente del gimnasio. ¿En qué puedo ayudarte?")
    else:
        respuesta = generar_respuesta(texto)
        enviar_respuesta(numero, respuesta)

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers.

\begin{equation} WhatsApp  \, \, \, \Rightarrow \, \, \, Configuración  \,\,de  \,\,la  \, \,API  \, \, \, \Rightarrow \, \, \, Configurar  \, \,webhooks \end{equation}

In [ ]:
# Installar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://wistful-discharge-upfront.ngrok-free.dev/webhook


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje para confirmar que el saludo inicial funciona, y un segundo mensaje para confirmar que ya se recibe una respuesta generada por Llama.

In [ ]:
# Mandar mensaje al bot para recibir un saludo

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215587654321", "text": {"body": "Hola"}}
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Primer mensaje:", r1.status_code, r1.text)

INFO:     127.0.0.1:39236 - "POST /webhook HTTP/1.1" 200 OK
Primer mensaje: 200 {"status":"ok"}


In [ ]:
# Seguir conversando con el bot para interactuar con Llama

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215587654321", "text": {"body": "¿A qué hora abren los sábados?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Segundo mensaje:", r2.status_code, r2.text)

# la respuesta demora aprox. de 1 a 2 min. al ejecutar local en colab

INFO:     127.0.0.1:47986 - "POST /webhook HTTP/1.1" 200 OK
Segundo mensaje: 200 {"status":"ok"}
